In [5]:
from datasets import load_dataset
dataset = load_dataset("tuetschek/multi_woz_v22")

Generating test split: 100%|██████████| 1000/1000 [00:00<00:00, 4350.57 examples/s]


In [7]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['dialogue_id', 'services', 'turns'],
        num_rows: 8437
    })
    validation: Dataset({
        features: ['dialogue_id', 'services', 'turns'],
        num_rows: 1000
    })
    test: Dataset({
        features: ['dialogue_id', 'services', 'turns'],
        num_rows: 1000
    })
})


In [10]:
hotel_train = dataset["train"].filter(lambda x: "hotel" in x["services"])
hotel_val = dataset["validation"].filter(lambda x: "hotel" in x["services"])
hotel_test = dataset["test"].filter(lambda x: "hotel" in x["services"])

print(f"hotel train dialogies: {len(hotel_train)}")
print(f"hotel val dialogies: {len(hotel_val)}")
print(f"hotel test dialogies: {len(hotel_test)}")

Filter: 100%|██████████| 1000/1000 [00:04<00:00, 225.93 examples/s]

hotel train dialogies: 3369
hotel val dialogies: 418
hotel test dialogies: 395


In [ ]:
#single dialogue
import json
sample = hotel_train[0]
print("Dialogue ID:", sample["dialogue_id"])
print("Services:", sample["services"])
print("Number of turns:", len(sample["turns"]["utterance"]))
print()
print("first 4")
for i in range(4):
    speaker = sample["turns"]["speaker"][i]
    text = sample["turns"]["utterance"][i]
    print(f"  {'USER' if speaker == 0 else 'SYSTEM'}: {text}")

Dialogue ID: PMUL4398.json
Services: ['restaurant', 'hotel']
Number of turns: 12

First 4 utterances:
  USER: i need a place to dine in the center thats expensive
  SYSTEM: I have several options for you; do you prefer African, Asian, or British food?
  USER: Any sort of food would be fine, as long as it is a bit expensive. Could I get the phone number for your recommendation?
  SYSTEM: There is an Afrian place named Bedouin in the centre. How does that sound?


In [14]:
# Correctly parse slot annotations
for i in range(6):
    speaker = sample["turns"]["speaker"][i]
    text = sample["turns"]["utterance"][i]
    frame = sample["turns"]["frames"][i]
    
    print(f"\n{'USER' if speaker == 0 else 'SYSTEM'}: {text}")
    
    services = frame["service"]
    for j, state in enumerate(frame["state"]):
        service = services[j]
        slot_names = state["slots_values"]["slots_values_name"]
        slot_lists = state["slots_values"]["slots_values_list"]
        
        if slot_names:
            slots = {name: vals[0] for name, vals in zip(slot_names, slot_lists)}
            print(f"  [{service}] {slots}")


USER: i need a place to dine in the center thats expensive
  [restaurant] {'restaurant-area': 'centre', 'restaurant-pricerange': 'expensive'}

SYSTEM: I have several options for you; do you prefer African, Asian, or British food?

USER: Any sort of food would be fine, as long as it is a bit expensive. Could I get the phone number for your recommendation?
  [restaurant] {'restaurant-area': 'centre', 'restaurant-pricerange': 'expensive'}

SYSTEM: There is an Afrian place named Bedouin in the centre. How does that sound?

USER: Sounds good, could I get that phone number? Also, could you recommend me an expensive hotel?
  [restaurant] {'restaurant-area': 'centre', 'restaurant-name': 'bedouin', 'restaurant-pricerange': 'expensive'}
  [hotel] {'hotel-pricerange': 'expensive', 'hotel-type': 'hotel'}

SYSTEM: Bedouin's phone is 01223367660. As far as hotels go, I recommend the University Arms Hotel in the center of town.
